# Colab Training via VS Code Extension

Make sure you have connected this notebook to a **Google Colab Kernel** using the top-right kernel selector.


In [1]:
# 1. Clone Codebase from Git
import os
import shutil

# Remove the old clone so we get the fresh code with the qm9.py fix
if os.path.exists("/content/project"):
    shutil.rmtree("/content/project")

REPO_URL = "https://github.com/KoyaTofu42/cfg-molecular-diffusion.git"

%cd /
!git clone $REPO_URL /content/project
%cd /content/project

/
Cloning into '/content/project'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 114 (delta 30), reused 107 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 5.28 MiB | 10.20 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/project


In [3]:
# 2. Install dependencies
!pip install torch torchvision torchaudio -q
!pip install matplotlib numpy scipy rdkit-pypi wandb -q

ERROR: Could not find a version that satisfies the requirement rdkit-pypi (from versions: none)
ERROR: No matching distribution found for rdkit-pypi


In [4]:
# 3. Authenticate with Weights & Biases (WandB)
import wandb
import os
import getpass

# Using getpass prompts you in VSCode securely without saving the key in the notebook
if "WANDB_API_KEY" not in os.environ:
    os.environ["WANDB_API_KEY"] = getpass.getpass("Enter your WANDB_API_KEY: ")

wandb.login()

WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "").strip()
if not WANDB_ENTITY:
    WANDB_ENTITY = input(
        "Enter your WandB entity/username (account name only): "
    ).strip()
WANDB_ENTITY = WANDB_ENTITY.split("/", 1)[0].strip()
if not WANDB_ENTITY:
    raise ValueError("WANDB_ENTITY is required")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rk406 (rk406-01) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# 4. (Optional) Restore Checkpoint from WandB
# If your Colab runtime disconnected and you want to resume training:
# Set WANDB_RUN_PATH to a real run path like 'username/e3_diffusion/run-12345'.

RUN_ID = "8q98b45b"  # Replace with your actual run ID
WANDB_RUN_PATH = f"rk406-01/e3_diffusion/runs/{RUN_ID}"
RESUME_ARGS = ""  # Leave blank if starting from scratch

if RUN_ID:
    api = wandb.Api()
    run = api.run(WANDB_RUN_PATH)
    os.makedirs(
        "/content/project/e3_diffusion_for_molecules/outputs/cfg_multi_property",
        exist_ok=True,
    )
    for file in run.files():
        if file.name.endswith(".npy") or file.name.endswith(".pickle"):
            file.download(
                root="/content/project/e3_diffusion_for_molecules/outputs/cfg_multi_property",
                replace=True,
            )
    print("Restored checkpoint from WandB")
    run_id = WANDB_RUN_PATH.split("/")[-1]
    RESUME_ARGS = f"--resume outputs/cfg_multi_property --wandb_run_id {run_id}"

Restored checkpoint from WandB


In [12]:
# 5. Download Dataset from WandB
import wandb
import os
import shutil

dataset_dir = "/content/project/e3_diffusion_for_molecules/qm9/temp/qm9"
if os.path.exists(dataset_dir):
    shutil.rmtree(dataset_dir)
os.makedirs(dataset_dir, exist_ok=True)

print("Downloading QM9 dataset from WandB Artifacts...")
api = wandb.Api()
artifact = api.artifact(f"{WANDB_ENTITY}/e3_diffusion/qm9_raw:latest", type="dataset")
downloaded_dir = artifact.download(root=dataset_dir)

expected_files = ["dsgdb9nsd.xyz.tar.bz2", "uncharacterized.txt", "atomref.txt"]
for filename in expected_files:
    source_path = os.path.join(downloaded_dir, filename)
    target_path = os.path.join(dataset_dir, filename)
    if os.path.exists(source_path) and source_path != target_path:
        shutil.move(source_path, target_path)

print("Dataset downloaded successfully!")
for filename in expected_files:
    print(os.path.exists(os.path.join(dataset_dir, filename)), filename)

wandb: Downloading large artifact 'qm9_raw:latest', 82.62MB. 3 files...
wandb:   3 of 3 files downloaded.  
Done. 00:00:00.6 (141.8MB/s)


Dataset downloaded successfully!
True dsgdb9nsd.xyz.tar.bz2
True uncharacterized.txt
True atomref.txt


In [ ]:
# 6. Start Training! (Optimized for Colab Free Tier)

!cd e3_diffusion_for_molecules && python main_qm9.py \
    --n_epochs 3000 \
    --exp_name cfg_multi_property \
    --conditioning alpha mu gap \
    --cfg_dropout_prob 0.15 \
    --cfg_dropout_mode joint \
    --n_stability_samples 100 \
    --diffusion_noise_schedule polynomial_2 \
    --diffusion_noise_precision 1e-5 \
    --diffusion_steps 500 \
    --diffusion_loss_type l2 \
    --batch_size 64 \
    --num_workers 2 \
    --nf 128 \
    --n_layers 6 \
    --lr 2e-4 \
    --normalize_factors "[1,4,1]" \
    --test_epochs 20 \
    --ema_decay 0.9999 \
    --save_model True \
    --wandb_usr "{WANDB_ENTITY}" \
    {RESUME_ARGS}

Namespace(exp_name='cfg_multi_property_resume', model='egnn_dynamics', probabilistic_model='diffusion', diffusion_steps=1000, diffusion_noise_schedule='polynomial_2', diffusion_noise_precision=1e-05, diffusion_loss_type='l2', n_epochs=3000, batch_size=32, lr=0.0001, brute_force=False, actnorm=True, break_train_epoch=False, dp=True, condition_time=True, clip_grad=True, trace='hutch', n_layers=6, inv_sublayers=1, nf=128, tanh=True, attention=True, norm_constant=1, sin_embedding=False, ode_regularization=0.001, dataset='qm9', datadir='qm9/temp', filter_n_atoms=None, dequantization='argmax_variational', n_report_steps=1, wandb_usr='rk406-01', no_wandb=False, online=True, no_cuda=False, save_model=True, generate_epochs=1, num_workers=0, test_epochs=50, data_augmentation=False, conditioning=['alpha', 'mu', 'gap'], resume='outputs/cfg_multi_property', start_epoch=0, ema_decay=0.9999, augment_noise=0, n_stability_samples=500, normalize_factors=[1, 4, 10], remove_h=False, include_charges=True, 

Epoch: 10, iter: 245/3125, Loss 2.28, NLL: 2.28, RegTerm: 0.0, GradNorm: 1.0
Epoch: 10, iter: 246/3125, Loss 2.43, NLL: 2.43, RegTerm: 0.0, GradNorm: 0.8
Epoch: 10, iter: 247/3125, Loss 2.42, NLL: 2.42, RegTerm: 0.0, GradNorm: 0.5
Epoch: 10, iter: 248/3125, Loss 2.66, NLL: 2.66, RegTerm: 0.0, GradNorm: 1.4
Epoch: 10, iter: 249/3125, Loss 2.67, NLL: 2.67, RegTerm: 0.0, GradNorm: 0.8
Epoch: 10, iter: 250/3125, Loss 2.28, NLL: 2.28, RegTerm: 0.0, GradNorm: 0.8
Epoch: 10, iter: 251/3125, Loss 2.41, NLL: 2.41, RegTerm: 0.0, GradNorm: 1.3
Epoch: 10, iter: 252/3125, Loss 2.56, NLL: 2.56, RegTerm: 0.0, GradNorm: 1.6
Epoch: 10, iter: 253/3125, Loss 2.78, NLL: 2.78, RegTerm: 0.0, GradNorm: 0.5
Epoch: 10, iter: 254/3125, Loss 2.51, NLL: 2.51, RegTerm: 0.0, GradNorm: 0.4
Epoch: 10, iter: 255/3125, Loss 2.32, NLL: 2.32, RegTerm: 0.0, GradNorm: 0.9
Epoch: 10, iter: 256/3125, Loss 2.70, NLL: 2.70, RegTerm: 0.0, GradNorm: 0.6
Epoch: 10, iter: 257/3125, Loss 2.56, NLL: 2.56, RegTerm: 0.0, GradNorm: 1.3

: 